# Stage 0 — Environment Setup & Baseline Benchmark

**Goal:** Load `google/gemma-3-1b-it`, run it on the FLORES-200 English→Swahili evaluation set (1,012 pairs), and record the BLEU and chrF++ scores.

This score is our regression checkpoint. Every subsequent training phase must match or exceed it.

**Target to beat (Swahili Gemma 1B):**
| Metric | Score |
|--------|-------|
| BLEU   | 27.6  |
| chrF++ | 56.8  |
| Efficiency (BLEU/B) | 27.6 |

> **Kaggle setup:** Enable GPU (T4 x1), enable Internet access in Settings before running.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q transformers accelerate datasets sacrebleu huggingface_hub bitsandbytes

## 2. Verify GPU and environment

In [ ]:
import torch
import transformers
import platform

print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)
print(f"Python          : {platform.python_version()}")
print(f"PyTorch         : {torch.__version__}")
print(f"Transformers    : {transformers.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory      : {total_mem:.1f} GB")
else:
    print("WARNING: No GPU found. Inference will be very slow on CPU.")
    print("Go to Settings → Accelerator → GPU T4 x1 and re-run.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device    : {DEVICE}")

## 3. Authenticate with Hugging Face

Gemma is a gated model. You need to:
1. Accept the Gemma licence at https://huggingface.co/google/gemma-3-1b-it
2. Create a HF token at https://huggingface.co/settings/tokens
3. Add it as a Kaggle secret named `HF_TOKEN` (Add-ons → Secrets)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication successful.")
except Exception as e:
    print(f"Auth failed: {e}")
    print("Add your HF token as a Kaggle secret named HF_TOKEN.")

## 4. Load the FLORES-200 evaluation set

In [ ]:
from datasets import load_dataset

# FLORES-200 uses language codes: eng_Latn for English, swh_Latn for Swahili
print("Loading FLORES-200 devtest split...")
flores_en = load_dataset("facebook/flores", "eng_Latn", split="devtest", trust_remote_code=True)
flores_sw = load_dataset("facebook/flores", "swh_Latn", split="devtest", trust_remote_code=True)

print(f"English sentences : {len(flores_en)}")
print(f"Swahili sentences : {len(flores_sw)}")

# Confirm alignment
assert len(flores_en) == len(flores_sw), "Sentence count mismatch!"
print(f"\nSample pair:")
print(f"  EN: {flores_en[0]['sentence']}")
print(f"  SW: {flores_sw[0]['sentence']}")

In [ ]:
# Extract sentence lists
english_sentences = flores_en["sentence"]
swahili_references = flores_sw["sentence"]

EVAL_SIZE = len(english_sentences)  # 1012 pairs — full devtest
print(f"Evaluating on {EVAL_SIZE} sentence pairs (full FLORES-200 devtest).")

## 5. Tokenizer fragmentation audit

Before loading the model, we audit how badly the default Gemma tokenizer fragments Swahili morphology. We measure **average tokens per word** on 200 Swahili reference sentences. This is our pre-CPT baseline to compare against after Phase 1.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "google/gemma-3-1b-it"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Audit on 200 Swahili reference sentences
audit_sentences = swahili_references[:200]
total_tokens = 0
total_words  = 0

worst_examples = []

for sent in audit_sentences:
    words  = sent.split()
    tokens = tokenizer.encode(sent, add_special_tokens=False)
    ratio  = len(tokens) / len(words) if words else 0
    total_tokens += len(tokens)
    total_words  += len(words)
    worst_examples.append((ratio, sent[:60]))

avg_ratio = total_tokens / total_words
worst_examples.sort(reverse=True)

print("\n" + "=" * 50)
print("TOKENIZER FRAGMENTATION AUDIT (pre-CPT)")
print("=" * 50)
print(f"Sentences audited       : {len(audit_sentences)}")
print(f"Total words             : {total_words}")
print(f"Total tokens produced   : {total_tokens}")
print(f"Avg tokens/word         : {avg_ratio:.3f}")
print(f"\nTarget after CPT        : < {avg_ratio:.2f} (any reduction is progress)")
print("\nMost fragmented sentences:")
for ratio, snip in worst_examples[:5]:
    print(f"  {ratio:.2f} tok/word  |  {snip}...")

# Single word deep-dive
test_words = ["tutawasiliana", "wanaopigana", "utekelezaji", "maendeleo", "uhusiano"]
print("\nSingle-word fragmentation examples:")
for w in test_words:
    toks = tokenizer.convert_ids_to_tokens(tokenizer.encode(w, add_special_tokens=False))
    print(f"  {w:20s} → {toks}  ({len(toks)} tokens)")

## 6. Load the base model

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization keeps the model within T4 VRAM (16 GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading model: {MODEL_ID}")
print("This may take 2–4 minutes on first load (downloading ~2.5 GB)...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()

# Parameter count
total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"\nModel loaded successfully.")
print(f"Total parameters : {total_params:.2f}B")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"VRAM allocated   : {allocated:.2f} GB")
    print(f"VRAM reserved    : {reserved:.2f} GB")

## 7. Define inference configuration

These are the generation parameters from the research spec. Fixed for all evaluation runs to ensure reproducibility.

In [ ]:
GENERATION_CONFIG = {
    "temperature"       : 0.3,    # Focus on high-probability tokens — reduces hallucination
    "top_p"             : 0.95,   # Nucleus sampling for natural flow
    "top_k"             : 64,     # Diverse vocabulary usage
    "max_new_tokens"    : 128,    # Optimised for sentence-level translation
    "repetition_penalty": 1.1,    # Prevent word loops in long-form output
    "do_sample"         : True,   # Enable balanced linguistic phrasing
}

def build_prompt(english_sentence: str) -> str:
    """
    Bilingual instruction prompt — English input, forced Swahili output.
    Mirrors the Swahili Gemma 1B constraint: accepts EN/SW input, outputs SW only.
    """
    return (
        "<start_of_turn>user\n"
        f"Tafsiri sentensi hii kwa Kiswahili:\n{english_sentence}\n"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )

# Sanity check on one example
test_prompt = build_prompt("The students are learning at school today.")
print("Example prompt:")
print(test_prompt)

## 8. Smoke test — translate 5 sentences

In [ ]:
def translate(sentence: str) -> str:
    prompt = build_prompt(sentence)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **GENERATION_CONFIG,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens (strip the prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("Smoke test — 5 sentence translations")
print("=" * 60)
for i in range(5):
    en  = english_sentences[i]
    ref = swahili_references[i]
    hyp = translate(en)
    print(f"[{i+1}] EN : {en}")
    print(f"    REF: {ref}")
    print(f"    HYP: {hyp}")
    print()

## 9. Full FLORES-200 evaluation

Runs all 1,012 sentence pairs and computes BLEU and chrF++.

> Estimated time: ~25–35 min on T4. A progress bar will track completion.

In [ ]:
from sacrebleu.metrics import BLEU, CHRF
import time

bleu_scorer = BLEU(effective_order=True)
chrf_scorer = CHRF(word_order=2)  # word_order=2 → chrF++

hypotheses = []
failed     = []

start_time = time.time()
checkpoint_interval = 100

print(f"Starting evaluation on {EVAL_SIZE} sentences...")
print("Checkpoint scores printed every 100 sentences.\n")

for idx in range(EVAL_SIZE):
    en = english_sentences[idx]
    try:
        hyp = translate(en)
        if not hyp:
            hyp = "[EMPTY]"
    except Exception as e:
        hyp = "[ERROR]"
        failed.append((idx, str(e)))

    hypotheses.append(hyp)

    # Checkpoint every N sentences
    if (idx + 1) % checkpoint_interval == 0:
        elapsed   = time.time() - start_time
        remaining = (elapsed / (idx + 1)) * (EVAL_SIZE - idx - 1)
        cp_bleu   = bleu_scorer.corpus_score(hypotheses, [swahili_references[:idx+1]]).score
        cp_chrf   = chrf_scorer.corpus_score(hypotheses, [swahili_references[:idx+1]]).score
        print(
            f"[{idx+1:4d}/{EVAL_SIZE}] "
            f"BLEU: {cp_bleu:.2f}  chrF++: {cp_chrf:.2f}  "
            f"Elapsed: {elapsed/60:.1f}m  ETA: {remaining/60:.1f}m"
        )

total_time = time.time() - start_time
print(f"\nEvaluation complete in {total_time/60:.1f} minutes.")
if failed:
    print(f"Failed sentences: {len(failed)} (see `failed` list for details)")

## 10. Final scores & efficiency report

In [ ]:
final_bleu = bleu_scorer.corpus_score(hypotheses, [swahili_references]).score
final_chrf = chrf_scorer.corpus_score(hypotheses, [swahili_references]).score

efficiency_ratio = final_bleu / total_params  # BLEU / parameters-in-billions

# Comparison table
BASELINE = {"model": "Swahili Gemma 1B", "bleu": 27.6, "chrf": 56.8, "params": 1.0}

print("\n" + "=" * 60)
print("STAGE 0 — BASELINE BENCHMARK RESULTS")
print("=" * 60)
print(f"Model             : {MODEL_ID}")
print(f"Eval sentences    : {EVAL_SIZE}")
print(f"Parameters        : {total_params:.2f}B")
print()
print(f"{'Metric':<20} {'This run':>12} {'Swahili Gemma 1B':>18} {'Delta':>8}")
print("-" * 62)
print(f"{'BLEU':<20} {final_bleu:>12.2f} {BASELINE['bleu']:>18.2f} {final_bleu - BASELINE['bleu']:>+8.2f}")
print(f"{'chrF++':<20} {final_chrf:>12.2f} {BASELINE['chrf']:>18.2f} {final_chrf - BASELINE['chrf']:>+8.2f}")
print(f"{'Efficiency (BLEU/B)':<20} {efficiency_ratio:>12.2f} {BASELINE['bleu']/BASELINE['params']:>18.2f} {efficiency_ratio - BASELINE['bleu']/BASELINE['params']:>+8.2f}")
print("=" * 62)

if final_bleu >= BASELINE["bleu"]:
    print(f"\nBase model already meets or exceeds the BLEU target before fine-tuning.")
    print(f"CPT + SFT should push this significantly higher.")
else:
    gap = BASELINE["bleu"] - final_bleu
    print(f"\nBase model is {gap:.2f} BLEU points below the fine-tuned target.")
    print(f"This is expected — CPT + SFT is designed to close this gap.")

## 11. Save results to disk

In [ ]:
import json
import os

output_dir = "/kaggle/working/stage0_results"
os.makedirs(output_dir, exist_ok=True)

# 1. Translations (for qualitative review)
translations_log = []
for i, (en, ref, hyp) in enumerate(zip(english_sentences, swahili_references, hypotheses)):
    translations_log.append({"id": i, "en": en, "ref": ref, "hyp": hyp})

with open(f"{output_dir}/translations.json", "w", encoding="utf-8") as f:
    json.dump(translations_log, f, ensure_ascii=False, indent=2)

# 2. Scores summary
scores = {
    "stage"            : 0,
    "model"            : MODEL_ID,
    "eval_set"         : "FLORES-200 devtest",
    "eval_size"        : EVAL_SIZE,
    "parameters_B"     : round(total_params, 4),
    "bleu"             : round(final_bleu, 4),
    "chrf_pp"          : round(final_chrf, 4),
    "efficiency_ratio" : round(efficiency_ratio, 4),
    "generation_config": GENERATION_CONFIG,
    "baseline_bleu"    : BASELINE["bleu"],
    "baseline_chrf"    : BASELINE["chrf"],
    "failed_count"     : len(failed),
}

with open(f"{output_dir}/stage0_scores.json", "w") as f:
    json.dump(scores, f, indent=2)

# 3. Tokenizer fragmentation baseline
frag_report = {
    "pre_cpt_avg_tokens_per_word": round(avg_ratio, 4),
    "sentences_audited"          : len(audit_sentences),
    "total_words"                : total_words,
    "total_tokens"               : total_tokens,
}

with open(f"{output_dir}/fragmentation_baseline.json", "w") as f:
    json.dump(frag_report, f, indent=2)

print(f"Results saved to {output_dir}/")
print(f"  translations.json          ({len(translations_log)} entries)")
print(f"  stage0_scores.json")
print(f"  fragmentation_baseline.json")
print("\nStage 0 complete. Download these files — they are the checkpoint for all future phases.")

## 12. Error analysis — qualitative review

Spot-check the 10 translations with the lowest sentence-level BLEU scores to understand where the base model fails before fine-tuning.

In [ ]:
# Sentence-level BLEU for each pair
sentence_bleus = []
for hyp, ref in zip(hypotheses, swahili_references):
    score = bleu_scorer.sentence_score(hyp, [ref]).score
    sentence_bleus.append(score)

# Sort by lowest score
indexed = sorted(enumerate(sentence_bleus), key=lambda x: x[1])

print("10 worst translations (lowest sentence BLEU):")
print("=" * 70)
for rank, (idx, score) in enumerate(indexed[:10], 1):
    print(f"[#{rank} | BLEU {score:.2f} | sentence {idx}]")
    print(f"  EN : {english_sentences[idx]}")
    print(f"  REF: {swahili_references[idx]}")
    print(f"  HYP: {hypotheses[idx]}")
    print()

print("\n10 best translations (highest sentence BLEU):")
print("=" * 70)
for rank, (idx, score) in enumerate(indexed[-10:][::-1], 1):
    print(f"[#{rank} | BLEU {score:.2f} | sentence {idx}]")
    print(f"  EN : {english_sentences[idx]}")
    print(f"  REF: {swahili_references[idx]}")
    print(f"  HYP: {hypotheses[idx]}")
    print()